# Exp 8: all-fish accumulation-evidence raster

This notebook is the maintained all-fish raster workflow for `Exp_8_accumulation_ev`. It keeps the reusable `src` helper flow from the Exp 1 notebook: load and validate the selected cohort, build an all-fish z-score raster, then render mean traces, left/right active-neuron diagnostics, motion-period counts, and lifetime sparseness.

The Exp 8 settings below were checked against the local experiment folders, four-block logs, trajectory timing, and the maintained several-fish configuration. Edit only `USER_SETTINGS` and `REPORT_SETTINGS`.


In [ ]:
# Cell 01 - Setup
from pathlib import Path
import sys
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.analysis_tools as at
import src.multifish_analysis as mfa
import src.plotting as plott
plott = importlib.reload(plott)
import src.reusable_several_fish as rsf
rsf = importlib.reload(rsf)


In [ ]:
# Cell 02 - Exp 8 settings
# IDs, cohort, blocks, and timing were validated from the local Exp 8 data/configuration.
USER_SETTINGS = {
    'experiment_name': 'Exp_8_accumulation_ev',
    'analysis_label': 'Exp 8 accumulation evidence',
    'data_base': Path(r'D:\Alejandro\Data'),
    # left = 10, 11, right = 4, 5, 7, both = 2, 3, 9
    # Maintained responsive Exp 8 cohort; L800_f01, f06, and f08 are excluded as bad-signal recordings.
    'fish_ids': ['L800_f02', 'L800_f03', 'L800_f09','L800_f04', 'L800_f05', 'L800_f07'],
    # Every trajectory occurs once per B1--B4 block (four repetitions per stimulus).
    'selected_blocks': [f'B{number}' for number in range(1, 5)],
    'verbose_loading': False,
    # 2 Hz acquisition and the 5 s / 25 s trial-alignment window are the maintained Exp 8 settings.
    # motion_onset_s is the canonical 8 s stimulus-program onset; exact per-trajectory markers come from the loader.
    'timing': {'fps_2p': 2.0, 't_pre_s': 5.0, 't_post_s': 30.0, 'motion_onset_s': 8.0, 'tau_s': 6.0},
    # Validated loader mapping, ordered left then right: base, segment 2, segment 1, random segment 1,
    # segments 1/3/5, rocking, five-position flicker, one-position flicker.
    #'stim_order': [5, 4, 3, 2, 1, 8, 7, 6, 16, 15, 14, 13, 12, 11, 10, 9],
    'stim_order': [5, 4, 3, 2, 1, 8, 7, 6, 16, 15, 14, 13, 12, 11, 10, 9],
    'combine_mode': 'mean',
    'trace_type': 'zscore',
    'raster_sort_mode':  'left_right_index',
    'left_control': 5,   # LeB
    'right_control': 16, # RiB
}

REPORT_SETTINGS = {
    'save_report': False,
    'run_label': 'exp8_accumulation_evidence_all_fish_raster',
    'report_format': 'html',
    'export_notebook': True,
    'fallback_to_html': True,
    'comments': 'Enable only after reviewing the Exp 8 figures.',
}


In [ ]:
# Cell 03 - Load the selected Exp 8 cohort and validate every configured stimulus ID.
raster_inputs = rsf.load_and_preflight_fish_raster_inputs(USER_SETTINGS)
globals().update(raster_inputs)
stimuli_colors, stimuli_linestyles = plott.build_stimulus_style_maps(stimuli_names=stimuli_names)
print('Detected Exp 8 stimulus map:', stimuli_id_map)


In [ ]:
# Cell 04 - All-fish z-score raster.
FIGSIZE = (13, 8)
raster_result = rsf.build_all_fish_raster_figure(
    all_fish_data, fish_ids, stim_order, reference_fish, timing, settings,
    stimuli_colors, stimuli_linestyles, figsize=FIGSIZE,
)
flat_matrix_all_fish = raster_result['flat_matrix_all_fish']
plt.show()


In [ ]:
# Cell 05 - Mean z-score traces in the configured Exp 8 stimulus order.
FIGSIZE = (10, 4)
mean_trace_result = rsf.build_plot_all_fish_mean_zscore_traces(
    all_fish_data, fish_ids, stim_order, reference_fish, timing,
    stimuli_colors, stimuli_linestyles, figsize=FIGSIZE,
)


In [ ]:
# Cell 06 - Left/right active-neuron overlap and significant-raster diagnostics.
OVERLAP_FIGSIZE = (5, 5)
RASTER_FIGSIZE = (16, 8)
OVERLAP_STIMULI = {
    'left': [5, 4, 3, 2, 1, 8, 7, 6],
    'right': [16, 15, 14, 12, 13, 11, 10, 9],
}
# Build shared overlap data first; draw rasters here from the reloaded plotting module.
overlap_results = rsf.plot_left_right_active_overlap_diagnostics(
    all_fish_data, fish_ids, reference_fish, timing, OVERLAP_STIMULI,
    motion_onset_s=timing['motion_onset_s'], tau_s=timing['tau_s'],
    show_overlap=True, show_significant_raster=False, overlap_figsize=OVERLAP_FIGSIZE,
    raster_figsize=RASTER_FIGSIZE,
)
for side in ('left', 'right'):
    diagnostic_data = overlap_results[side]
    diagnostic = diagnostic_data['active_trace_diagnostic']
    if diagnostic['trace_matrix'].shape[0] == 0:
        print(f'No active neurons passed the {side} settings.')
        continue
    plott.plot_active_trace_decision_diagnostic(
        diagnostic,
        fps_2p=timing['fps_2p'],
        stimuli_durations=reference_fish['stimuli_durations'],
        t_pre_s=timing['t_pre_s'],
        sort_mode=None,
        neuron_order=diagnostic_data['diagnostic_neuron_order'],
        trace_cmap='Greys',
        trace_title=f'{side.capitalize()} significant raster',
        show_active_count_trace=True,
        active_count_threshold=0.5,
        active_count_color='black',
        active_count_ylabel='# Active neurons',
        active_count_height_ratio=2.4,
        figsize=RASTER_FIGSIZE,
    )
    plt.show()


In [ ]:
# Cell 09 - Motion-period active neurons per stimulus; each black point is one fish.
# The helper deliberately creates no fish legend, and receives this settings-cell order unchanged.
FIGSIZE = (5, 3.5)
motion_active_result = rsf.plot_motion_active_neuron_counts(
    all_fish_data, fish_ids, reference_fish, timing, stim_order, stimuli_colors,
    figsize=FIGSIZE, motion_onset_s=timing['motion_onset_s'], tau_s=timing['tau_s'],
)
active_neuron_counts = motion_active_result['counts']


## Per-fish active-neuron AUC decomposition

For `RiBseg1 ? RiBseg2` and `RiBseg2 ? RiB`, this section reuses Cell 09's strict motion-period active-neuron decisions and Cell 08's per-neuron z-score AUC. Raw AUC is **total integrated z-score activity** over each stimulus-specific response window; duration-normalized AUC divides each neuron's AUC by that window's integrated time span. Percent contributions are expressed relative to the corresponding total change.


In [ ]:
# Cell 10 - Per-fish AUC decomposition from Cell 09 activity decisions and Cell 08 AUCs.
ACCUMULATION_AUC_PAIRS = [('RiBseg1', 'RiBseg2'), ('RiBseg2', 'RiB')]
COMPONENT_ORDER = ['Shared-neuron gain', 'Recruited neurons', 'Lost neurons']

accumulation_auc_rows = []
for reference_label, larger_label in ACCUMULATION_AUC_PAIRS:
    pair_selection = at.resolve_selected_stimuli(
        [reference_label, larger_label], stimuli_id_map=reference_fish['stimuli_id_map'],
        available_stimuli=reference_fish['trial_aligned_traces_z_core'].keys(),
    )
    reference_id, larger_id = pair_selection['stimulus_ids']
    reference_label, larger_label = pair_selection['stimulus_labels']
    comparison_label = f'{reference_label} -> {larger_label}'
    for fish_id in fish_ids:
        fish = all_fish_data[fish_id]
        active = motion_active_result['active_matrices'][fish_id]
        reference_active = active[reference_id].to_numpy(dtype=bool)
        larger_active = active[larger_id].to_numpy(dtype=bool)
        shared = reference_active & larger_active
        recruited = ~reference_active & larger_active
        lost = reference_active & ~larger_active
        auc_matrix = mfa.build_zscore_response_matrix_for_fish(
            fish['trial_aligned_traces_z_core'], pair_selection['stimulus_ids'], fish['stimuli_durations'],
            stimuli_id_map=fish['stimuli_id_map'], kept_neuron_indices=fish['kept_neuron_indices'],
            fps_2p=timing['fps_2p'], t_pre_s=timing['t_pre_s'],
            motion_onset_s=timing['motion_onset_s'], tau_s=timing['tau_s'],
        )
        durations_s = {}
        for stimulus_id, stimulus_label in zip(pair_selection['stimulus_ids'], pair_selection['stimulus_labels']):
            trace_key = stimulus_id if stimulus_id in fish['trial_aligned_traces_z_core'] else str(stimulus_id)
            window = at.compute_response_window_frames(
                fish['trial_aligned_traces_z_core'][trace_key].shape[1], fps_2p=timing['fps_2p'],
                t_pre_s=timing['t_pre_s'], motion_onset_s=timing['motion_onset_s'], stimulus=stimulus_id,
                stimuli_durations=fish['stimuli_durations'], stimuli_id_map=fish['stimuli_id_map'], tau_s=timing['tau_s'],
            )
            response_time_s = window['time_s'][window['frame_indices']]
            durations_s[stimulus_label] = max(float(np.ptp(response_time_s)), 1 / timing['fps_2p'])
        for activity_measure, scale in [
            ('Raw AUC (integrated activity)', {reference_label: 1.0, larger_label: 1.0}),
            ('Duration-normalized AUC', {label: 1 / duration for label, duration in durations_s.items()}),
        ]:
            auc_reference = auc_matrix[reference_label].to_numpy() * scale[reference_label]
            auc_larger = auc_matrix[larger_label].to_numpy() * scale[larger_label]
            total_reference = auc_reference[reference_active].sum()
            total_larger = auc_larger[larger_active].sum()
            delta_total = total_larger - total_reference
            components = {
                'Shared-neuron gain': (auc_larger[shared] - auc_reference[shared]).sum(),
                'Recruited neurons': auc_larger[recruited].sum(),
                'Lost neurons': -auc_reference[lost].sum(),
            }
            component_sum = sum(components.values())
            if not np.isclose(delta_total, component_sum, rtol=1e-8, atol=1e-10):
                raise AssertionError(f'{fish_id}, {comparison_label}, {activity_measure}: AUC decomposition failed.')
            for component, contribution in components.items():
                accumulation_auc_rows.append({
                    'fish_id': fish_id, 'comparison': comparison_label, 'activity_measure': activity_measure,
                    'component': component, 'contribution': contribution,
                    'percent_of_delta_total': np.nan if np.isclose(delta_total, 0) else 100 * contribution / delta_total,
                    'total_ref': total_reference, 'total_large': total_larger, 'delta_total': delta_total,
                    'shared_gain': components['Shared-neuron gain'], 'recruitment': components['Recruited neurons'],
                    'loss': components['Lost neurons'], 'n_recruited': int(recruited.sum()), 'n_lost': int(lost.sum()),
                    'mean_shared_auc_change': np.nan if not shared.any() else (auc_larger[shared] - auc_reference[shared]).mean(),
                    'reference_window_s': durations_s[reference_label], 'larger_window_s': durations_s[larger_label],
                    'decomposition_verified': True,
                })

auc_decomposition = pd.DataFrame(accumulation_auc_rows)
auc_decomposition_summary = auc_decomposition.drop_duplicates(['fish_id', 'comparison', 'activity_measure'])[
    ['fish_id', 'comparison', 'activity_measure', 'total_ref', 'total_large', 'delta_total', 'shared_gain',
     'recruitment', 'loss', 'n_recruited', 'n_lost', 'mean_shared_auc_change', 'reference_window_s',
     'larger_window_s', 'decomposition_verified']]
display(auc_decomposition_summary)


In [ ]:
# Cell 11 - Per-fish raw-AUC decomposition summary (two figures only).
RAW_AUC_LABEL = 'Raw AUC (integrated activity)'
raw_auc_summary = auc_decomposition_summary.query('activity_measure == @RAW_AUC_LABEL').copy()

# Figure 1 - AUC decomposition: group mean bars plus one dot per fish.
fig, axes = plt.subplots(1, len(ACCUMULATION_AUC_PAIRS), figsize=(15, 5), sharey=True, squeeze=False)
for ax, (reference_label, larger_label) in zip(axes[0], ACCUMULATION_AUC_PAIRS):
    comparison = f'{reference_label} -> {larger_label}'
    values = raw_auc_summary[raw_auc_summary['comparison'] == comparison].set_index('fish_id').reindex(fish_ids)
    # These are additive population-AUC contributions, not per-neuron means.
    # For every fish: delta_total = shared_gain + recruitment + loss.
    auc_metrics = {
        'Total AUC increase': values['delta_total'],
        'Shared-neuron AUC change': values['shared_gain'],
        f'Recruited in {larger_label}': values['recruitment'],
        f'Lost from {reference_label}': values['loss'],
    }
    x = np.arange(len(auc_metrics))
    for position, metric_values in enumerate(auc_metrics.values()):
        ax.bar(position, metric_values.mean(), color='0.80', edgecolor='0.35', zorder=1)
        ax.scatter(np.full(len(fish_ids), position), metric_values, color='black', s=28, alpha=0.8, zorder=2)
    ax.axhline(0, color='0.5', linewidth=0.8, zorder=0)
    ax.set(xticks=x, xticklabels=list(auc_metrics), title=comparison)
    ax.tick_params(axis='x', rotation=25)
    ax.spines[['top', 'right']].set_visible(False)
axes[0, 0].set_ylabel('Raw z-score AUC contribution (integrated activity)')
fig.suptitle('AUC increase decomposition: total = shared change + recruitment + loss', y=1.02)
fig.tight_layout()
plt.show()

# Figure 2 - Neuron numbers: group mean bars plus one dot per fish.
fig, axes = plt.subplots(1, len(ACCUMULATION_AUC_PAIRS), figsize=(10, 4.5), sharey=True, squeeze=False)
for ax, (reference_label, larger_label) in zip(axes[0], ACCUMULATION_AUC_PAIRS):
    comparison = f'{reference_label} -> {larger_label}'
    values = raw_auc_summary[raw_auc_summary['comparison'] == comparison].set_index('fish_id').reindex(fish_ids)
    count_metrics = {f'Recruited in {larger_label}': values['n_recruited'], f'Lost from {reference_label}': values['n_lost']}
    x = np.arange(len(count_metrics))
    for position, metric_values in enumerate(count_metrics.values()):
        ax.bar(position, metric_values.mean(), color='0.80', edgecolor='0.35', zorder=1)
        ax.scatter(np.full(len(fish_ids), position), metric_values, color='black', s=28, alpha=0.8, zorder=2)
    ax.set(xticks=x, xticklabels=list(count_metrics), title=comparison)
    ax.tick_params(axis='x', rotation=25)
    ax.spines[['top', 'right']].set_visible(False)
axes[0, 0].set_ylabel('Neuron count')
fig.suptitle('Active-neuron recruitment and loss: group mean bars and individual fish', y=1.02)
fig.tight_layout()
plt.show()


In [ ]:
# Cell 12 - Population decomposition: active-neuron number versus mean active-neuron AUC.
population_decomposition_rows = []
for reference_label, larger_label in ACCUMULATION_AUC_PAIRS:
    pair_selection = at.resolve_selected_stimuli(
        [reference_label, larger_label], stimuli_id_map=reference_fish['stimuli_id_map'],
        available_stimuli=reference_fish['trial_aligned_traces_z_core'].keys(),
    )
    reference_id, larger_id = pair_selection['stimulus_ids']
    reference_label, larger_label = pair_selection['stimulus_labels']
    comparison = f'{reference_label} -> {larger_label}'
    for fish_id in fish_ids:
        active = motion_active_result['active_matrices'][fish_id]
        n_reference = int(active[reference_id].sum())
        n_larger = int(active[larger_id].sum())
        auc_totals = raw_auc_summary[(raw_auc_summary['fish_id'] == fish_id) & (raw_auc_summary['comparison'] == comparison)].iloc[0]
        mean_auc_reference = np.nan if n_reference == 0 else auc_totals['total_ref'] / n_reference
        mean_auc_larger = np.nan if n_larger == 0 else auc_totals['total_large'] / n_larger
        d_n = n_larger - n_reference
        d_a = mean_auc_larger - mean_auc_reference
        number_effect = d_n * mean_auc_reference
        mean_auc_effect = n_reference * d_a
        interaction = d_n * d_a
        delta_r = auc_totals['total_large'] - auc_totals['total_ref']
        if not np.isclose(delta_r, number_effect + mean_auc_effect + interaction, rtol=1e-8, atol=1e-10):
            raise AssertionError(f'{fish_id}, {comparison}: population decomposition failed.')
        population_decomposition_rows.append({
            'fish_id': fish_id, 'comparison': comparison, 'n_ref': n_reference, 'n_large': n_larger,
            'mean_auc_ref': mean_auc_reference, 'mean_auc_large': mean_auc_larger,
            'delta_R': delta_r, 'number_effect': number_effect, 'mean_auc_effect': mean_auc_effect,
            'interaction': interaction, 'decomposition_verified': True,
        })

population_decomposition = pd.DataFrame(population_decomposition_rows)
display(population_decomposition)

fig, axes = plt.subplots(1, len(ACCUMULATION_AUC_PAIRS), figsize=(12, 4.5), sharey=True, squeeze=False)
for ax, (reference_label, larger_label) in zip(axes[0], ACCUMULATION_AUC_PAIRS):
    comparison = f'{reference_label} -> {larger_label}'
    values = population_decomposition[population_decomposition['comparison'] == comparison].set_index('fish_id').reindex(fish_ids)
    effects = {'Number effect': values['number_effect'], 'Mean-AUC effect': values['mean_auc_effect'], 'Interaction': values['interaction']}
    x = np.arange(len(effects))
    for position, effect_values in enumerate(effects.values()):
        ax.bar(position, effect_values.mean(), color='0.80', edgecolor='0.35', zorder=1)
        ax.scatter(np.full(len(fish_ids), position), effect_values, color='black', s=28, alpha=0.8, zorder=2)
    ax.axhline(0, color='0.5', linewidth=0.8, zorder=0)
    ax.set(xticks=x, xticklabels=list(effects), title=comparison)
    ax.tick_params(axis='x', rotation=25)
    ax.spines[['top', 'right']].set_visible(False)
axes[0, 0].set_ylabel('Contribution to delta population activity (raw z-score AUC)')
fig.suptitle('Population decomposition: active-neuron number, mean response, and interaction', y=1.02)
fig.tight_layout()
plt.show()


In [ ]:
# Cell 08 - Lifetime sparseness and the high-sparseness z-score raster.
SPARSENESS_FIGSIZE = (7, 4.5)
RASTER_FIGSIZE = (13, 8)
SPARSENESS_STIMULI = stim_order
HIGH_LIFETIME_SPARSENESS = 0.75
HIGH_SPARSENESS_COMBINE_MODE = 'mean'
sparseness_result = rsf.plot_lifetime_sparseness_analysis(
    all_fish_data, fish_ids, reference_fish, timing, SPARSENESS_STIMULI,
    HIGH_LIFETIME_SPARSENESS, stimuli_colors, stimuli_linestyles,
    combine_mode=HIGH_SPARSENESS_COMBINE_MODE, sparseness_figsize=SPARSENESS_FIGSIZE,
    raster_figsize=RASTER_FIGSIZE, analysis_label=settings['analysis_label'],
)


In [ ]:
# Cell 09 - Optional reviewed-run report.
# Save this notebook before enabling REPORT_SETTINGS['save_report'] so the HTML includes current outputs.
notebook_path = repo_root / 'notebooks' / 'calcium' / 'exp_08_accumulation_ev' / '01_all_fish_raster.ipynb'
report_summary = pd.DataFrame([
    {
        'fish_id': fish_id,
        'kept_neurons': len(fish_data['kept_neuron_indices']),
        'detected_stimuli': len(fish_data['stimuli_ids']),
    }
    for fish_id, fish_data in all_fish_data.items()
])
report_result = rsf.save_analysis_report_run(
    settings=settings,
    report_settings=REPORT_SETTINGS,
    analysis_path=analysis_path,
    tables={'run_summary': report_summary, 'motion_active_neuron_counts': active_neuron_counts},
    notebook_path=notebook_path,
    report_name='01_all_fish_raster_exp8_report',
    extra_metadata={
        'stim_order': stim_order,
        'combine_mode': settings['combine_mode'],
        'trace_type': settings['trace_type'],
        'raster_sort_mode': settings['raster_sort_mode'],
        'all_fish_matrix_shape': flat_matrix_all_fish.shape,
    },
)
report_result
